# GridironIQ · What the cap says about winning

Using the cleaned 2011–2024 cap data (see [`01_cap_eda.ipynb`](01_cap_eda.ipynb)), this notebook presents the findings — every figure comes straight from the `gridiron.modeling` package, nothing is re-implemented here.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_curve, auc
from sklearn.model_selection import StratifiedKFold, cross_val_predict

from gridiron.modeling import cap_efficiency, clustering, ml_pipeline

sns.set_theme(style="whitegrid", palette="deep")

## 1. Do champions over- or under-pay by position?

The headline. `champion_premium` compares Super Bowl winners' positional cap share against the league average (2011–2024).

In [ ]:
prem = cap_efficiency.champion_premium().sort_values("premium")
colors = ["#cf222e" if v < 0 else "#1a7f37" for v in prem["premium"]]

fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(prem.index, prem["premium"] * 100, color=colors)
ax.axvline(0, color="k", lw=1)
ax.set(xlabel="champion cap share  minus  league average  (points)",
       title="Champions pay UP at TE/QB/EDGE and win paying LESS at RB/LB/CB")
plt.tight_layout()
prem.assign(premium=(prem["premium"] * 100).round(2))

And the win-correlation verdict (`efficiency_verdict`) — which positions' spending actually tracks winning, with a plain-English call:

In [ ]:
cap_efficiency.efficiency_verdict()

## 2. Roster archetypes (k-means on spending profiles)

Cluster every team-season by *how* it splits its cap, then score each archetype by how often it reaches and wins the Super Bowl.

In [ ]:
res = clustering.cluster_archetypes(k=5)
res.success.sort_values("sb_appearance_rate", ascending=False)

In [ ]:
prof = res.profiles.drop(columns=["label"])
fig, ax = plt.subplots(figsize=(11, 4))
sns.heatmap(prof, annot=True, fmt=".2f", cmap="YlGnBu",
            cbar_kws={"label": "mean cap share"}, ax=ax)
ax.set(title="Spending archetypes (k-means centroids)",
       xlabel="position group", ylabel="archetype")
plt.tight_layout()
res.profiles

## 3. Can positional spending predict a Super Bowl run?

A gut check on signal, not a betting model. `ml_pipeline` cross-validates a random forest on the positional-share features; here I draw the honest **out-of-fold** ROC and the permutation feature importances.

In [ ]:
X, y = ml_pipeline.get_xy(target="sb_appearance")
pipe = ml_pipeline.build_pipeline("rf")
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
proba = cross_val_predict(pipe, X, y, cv=cv, method="predict_proba")[:, 1]
fpr, tpr, _ = roc_curve(y, proba)

fig, ax = plt.subplots(figsize=(6, 6))
ax.plot(fpr, tpr, lw=2, label=f"random forest  (AUC = {auc(fpr, tpr):.2f})")
ax.plot([0, 1], [0, 1], "k--", lw=1, label="chance")
ax.set(xlabel="false positive rate", ylabel="true positive rate",
       title="Out-of-fold ROC — SB appearance from positional cap shares")
ax.legend(loc="lower right")
plt.tight_layout()

In [ ]:
report = ml_pipeline.train_and_report(target="sb_appearance", model="rf")
print(f"5-fold CV AUC: {report.cv_auc_mean:.3f} +/- {report.cv_auc_std:.3f}  "
      f"(n={report.n_samples}, positives={report.n_positive})")

imp = report.importances.head(8).iloc[::-1]
fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(imp["feature"].str.replace("pct_", ""), imp["importance"], xerr=imp["std"])
ax.set(xlabel="permutation importance (drop in AUC)",
       title="Which positions move the model")
plt.tight_layout()
report.importances

## Takeaways

- Champions **pay up** at TE/QB/EDGE and win **spending less** at RB/LB/CB — the thesis, quantified.
- Spending *shape* clusters into archetypes with very different title rates.
- Positional cap shares carry real (if modest) signal for a Super Bowl run (~0.7 out-of-fold AUC) — cap allocation matters, but it isn't destiny.

All figures come straight from the `gridiron.modeling` package — the same code the Streamlit app and CLI use.